# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SamarBabar02/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*



### Rule

I will prioritize content pages that have low CTR but relatively strong search visibility. These pages already appear in strong search positions but receive relatively few clicks compared with their impressions. This makes them possible CTR improvement opportunities.

### Reason Code

- `LOW_CTR_STRONG_POSITION` — Low CTR with a strong search position (1–3).

### Action Label

- `REVIEW_CTR` — Review the page for a possible CTR improvement opportunity.

In [1]:
!pip -q install duckdb huggingface_hub

In [2]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

print("Hugging Face secret configured successfully.")

Hugging Face secret configured successfully.


In [3]:
rel = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
"""

In [4]:
query = f"""
SELECT *
FROM {rel}
WHERE DATE_TRUNC('month', report_date) = DATE '2026-03-01'
"""

df = con.sql(query).df()

print("Shape:", df.shape)
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Shape: (9841378, 31)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [5]:
print(df.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [6]:
print("Start date:", df["report_date"].min())
print("End date:", df["report_date"].max())
print("Total rows:", len(df))

Start date: 2026-03-01 00:00:00
End date: 2026-03-31 00:00:00
Total rows: 9841378


In [7]:
df[['gsc_impressions',
    'gsc_clicks',
    'gsc_avg_position',
    'ga4_pageviews',
    'ga4_sessions',
    'ga4_engaged_sessions',
    'ga4_total_engagement_sec',
    'scroll_events']].describe()

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_engaged_sessions,ga4_total_engagement_sec,scroll_events
count,9.841378e+06,9.841378e+06,3.611061e+06,6822637.0,6822637.0,6822637.0,6822637.0,6822637.0
mean,2.851812e+01,8.350782e-02,1.582665e+01,0.217637,0.190514,0.004331,0.698905,0.032261
std,1.559266e+02,7.814341e-01,1.985603e+01,2.142851,1.96875,0.076647,17.206612,0.413166
min,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0,0.0
25%,0.000000e+00,0.000000e+00,3.742120e+00,0.0,0.0,0.0,0.0,0.0
50%,0.000000e+00,0.000000e+00,7.500000e+00,0.0,0.0,0.0,0.0,0.0
75%,6.000000e+00,0.000000e+00,2.020000e+01,0.0,0.0,0.0,0.0,0.0
max,4.008400e+04,2.740000e+02,4.980000e+02,875.0,792.0,21.0,7083.0,254.0


In [8]:
df[['gsc_impressions',
    'gsc_clicks',
    'gsc_avg_position',
    'ga4_pageviews',
    'ga4_sessions',
    'ga4_engaged_sessions',
    'ga4_total_engagement_sec',
    'scroll_events']].isnull().mean().sort_values(ascending=False)

,0
gsc_avg_position,0.633074
ga4_pageviews,0.306740
ga4_engaged_sessions,0.306740
ga4_sessions,0.306740
ga4_total_engagement_sec,0.306740
scroll_events,0.306740
gsc_impressions,0.000000
gsc_clicks,0.000000


In [11]:
# Calculate CTR only where impressions are greater than 0
import numpy as np
import pandas as pd
df['ctr'] = np.where(
    df['gsc_impressions'] > 0,
    df['gsc_clicks'] / df['gsc_impressions'],
    np.nan
)

# Bucket CTR
df['ctr_bucket'] = pd.cut(
    df['ctr'],
    bins=[-np.inf, 0.02, 0.05, np.inf],
    labels=['Low', 'Medium', 'High']
)

# Bucket average position
df['position_bucket'] = pd.cut(
    df['gsc_avg_position'],
    bins=[-np.inf, 3, 10, np.inf],
    labels=['Strong (1-3)', 'Moderate (4-10)', 'Weak (>10)']
)

# CTR bucket table
ctr_table = (
    df.groupby('ctr_bucket', observed=False)
      .size()
      .reset_index(name='n')
)

print("CTR Bucket Table")
display(ctr_table)

# Position bucket table
position_table = (
    df.groupby('position_bucket', observed=False)
      .size()
      .reset_index(name='n')
)

print("\nPosition Bucket Table")
display(position_table)

CTR Bucket Table


,ctr_bucket,n
0,Low,3510885
1,Medium,65464
2,High,34712



Position Bucket Table


,position_bucket,n
0,Strong (1-3),727362
1,Moderate (4-10),1456122
2,Weak (>10),1427577


In [12]:
# Check CTR vs position together

ctr_position_table = pd.crosstab(
    df['position_bucket'],
    df['ctr_bucket'],
    margins=True
)

display(ctr_position_table)

ctr_bucket,Low,Medium,High,All
position_bucket,,,,
Strong (1-3),703452,14960,8950,727362
Moderate (4-10),1405902,33983,16237,1456122
Weak (>10),1401531,16521,9525,1427577
All,3510885,65464,34712,3611061


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [13]:
# Section 2 — Build the ranked queue

# Create a baseline score
# Lower CTR = higher priority
# Better search position = higher priority

df['baseline_score'] = (
    (1 - df['ctr'].clip(0, 1)) * 0.6
    + (1 / (1 + df['gsc_avg_position'])) * 0.4
)

# Create reason code
df['reason_code'] = np.where(
    (df['ctr_bucket'] == 'Low') &
    (df['position_bucket'] == 'Strong (1-3)'),
    'LOW_CTR_STRONG_POSITION',
    'OTHER'
)

# Create action label
df['action'] = np.where(
    df['reason_code'] == 'LOW_CTR_STRONG_POSITION',
    'REVIEW_CTR',
    'NO_ACTION'
)

# Rank pages by baseline score
ranked_queue = df[
    [
        'report_date',
        'client_hash_id',
        'content_hash_id',
        'gsc_impressions',
        'gsc_clicks',
        'ctr',
        'gsc_avg_position',
        'position_bucket',
        'baseline_score',
        'reason_code',
        'action'
    ]
].copy()

ranked_queue = ranked_queue.sort_values(
    'baseline_score',
    ascending=False
).reset_index(drop=True)

# Add rank
ranked_queue['rank'] = ranked_queue.index + 1

# Reorder columns
ranked_queue = ranked_queue[
    [
        'rank',
        'report_date',
        'client_hash_id',
        'content_hash_id',
        'gsc_impressions',
        'gsc_clicks',
        'ctr',
        'gsc_avg_position',
        'position_bucket',
        'baseline_score',
        'reason_code',
        'action'
    ]
]

# Create output directory
import os
os.makedirs('work/outputs', exist_ok=True)

# Write ranked queue
output_path = 'work/outputs/baseline_action_score.csv'
ranked_queue.to_csv(output_path, index=False)

print("Ranked queue created successfully.")
print("Rows:", len(ranked_queue))
print("Output:", output_path)

# Show top 20
display(ranked_queue.head(20))

Ranked queue created successfully.
Rows: 9841378
Output: work/outputs/baseline_action_score.csv


,rank,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,position_bucket,baseline_score,reason_code,action
0,1,2026-03-16,client_73cda7b4e4f265ea,content_34280304abbab05e,3,0,0.0,0.0,Strong (1-3),1.0,LOW_CTR_STRONG_POSITION,REVIEW_CTR
1,2,2026-03-16,client_73cda7b4e4f265ea,content_ea60d7cef612b29c,1,0,0.0,0.0,Strong (1-3),1.0,LOW_CTR_STRONG_POSITION,REVIEW_CTR
2,3,2026-03-01,client_73cda7b4e4f265ea,content_03940665a88bf663,1,0,0.0,0.0,Strong (1-3),1.0,LOW_CTR_STRONG_POSITION,REVIEW_CTR
3,4,2026-03-16,client_73cda7b4e4f265ea,content_05f82b2bad6817cc,1,0,0.0,0.0,Strong (1-3),1.0,LOW_CTR_STRONG_POSITION,REVIEW_CTR
4,5,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.0,0.0,Strong (1-3),1.0,LOW_CTR_STRONG_POSITION,REVIEW_CTR
5,6,2026-03-01,client_73cda7b4e4f265ea,content_a36aa972b8edda8f,2,0,0.0,0.0,Strong (1-3),1.0,LOW_CTR_STRONG_POSITION,REVIEW_CTR
6,7,2026-03-01,client_73cda7b4e4f265ea,content_d720dde3701523c0,1,0,0.0,0.0,Strong (1-3),1.0,LOW_CTR_STRONG_POSITION,REVIEW_CTR
7,8,2026-03-16,client_73cda7b4e4f265ea,content_eee959b8c3faf5c2,1,0,0.0,0.0,Strong (1-3),1.0,LOW_CTR_STRONG_POSITION,REVIEW_CTR
8,9,2026-03-01,client_73cda7b4e4f265ea,content_1f380a642aed423b,1,0,0.0,0.0,Strong (1-3),1.0,LOW_CTR_STRONG_POSITION,REVIEW_CTR
9,10,2026-03-05,client_73cda7b4e4f265ea,content_617fdc3f3a5d41eb,2,0,0.0,0.0,Strong (1-3),1.0,LOW_CTR_STRONG_POSITION,REVIEW_CTR


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [15]:
# 3. Top-20 Review

top20 = ranked_queue.head(20).copy()

def confidence_note(row):
    impressions = row["gsc_impressions"]

    if impressions <= 2:
        return "Very low evidence because impressions are extremely limited."
    elif impressions <= 10:
        return "Low confidence because the page has relatively few impressions."
    elif impressions <= 50:
        return "Moderate confidence; the signal is based on a limited number of impressions."
    else:
        return "Higher confidence because the signal is supported by more impressions."


def what_would_make_it_wrong(row):
    impressions = row["gsc_impressions"]

    if impressions <= 2:
        return "More impressions or clicks could change the observed CTR substantially."
    elif impressions <= 10:
        return "Additional search impressions and clicks could change the CTR signal."
    elif impressions <= 50:
        return "A larger observation window could change the measured CTR."
    else:
        return "A longer observation window or additional context could change the decision."


top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(
    what_would_make_it_wrong, axis=1
)

# Display the Top-20 review
display(
    top20[
        [
            "rank",
            "content_hash_id",
            "gsc_impressions",
            "gsc_clicks",
            "ctr",
            "gsc_avg_position",
            "position_bucket",
            "baseline_score",
            "action",
            "reason_code",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ]
)

,rank,content_hash_id,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,position_bucket,baseline_score,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_34280304abbab05e,3,0,0.0,0.0,Strong (1-3),1.0,REVIEW_CTR,LOW_CTR_STRONG_POSITION,Low confidence because the page has relatively...,Additional search impressions and clicks could...
1,2,content_ea60d7cef612b29c,1,0,0.0,0.0,Strong (1-3),1.0,REVIEW_CTR,LOW_CTR_STRONG_POSITION,Very low evidence because impressions are extr...,More impressions or clicks could change the ob...
2,3,content_03940665a88bf663,1,0,0.0,0.0,Strong (1-3),1.0,REVIEW_CTR,LOW_CTR_STRONG_POSITION,Very low evidence because impressions are extr...,More impressions or clicks could change the ob...
3,4,content_05f82b2bad6817cc,1,0,0.0,0.0,Strong (1-3),1.0,REVIEW_CTR,LOW_CTR_STRONG_POSITION,Very low evidence because impressions are extr...,More impressions or clicks could change the ob...
4,5,content_05597932fe4da067,1,0,0.0,0.0,Strong (1-3),1.0,REVIEW_CTR,LOW_CTR_STRONG_POSITION,Very low evidence because impressions are extr...,More impressions or clicks could change the ob...
5,6,content_a36aa972b8edda8f,2,0,0.0,0.0,Strong (1-3),1.0,REVIEW_CTR,LOW_CTR_STRONG_POSITION,Very low evidence because impressions are extr...,More impressions or clicks could change the ob...
6,7,content_d720dde3701523c0,1,0,0.0,0.0,Strong (1-3),1.0,REVIEW_CTR,LOW_CTR_STRONG_POSITION,Very low evidence because impressions are extr...,More impressions or clicks could change the ob...
7,8,content_eee959b8c3faf5c2,1,0,0.0,0.0,Strong (1-3),1.0,REVIEW_CTR,LOW_CTR_STRONG_POSITION,Very low evidence because impressions are extr...,More impressions or clicks could change the ob...
8,9,content_1f380a642aed423b,1,0,0.0,0.0,Strong (1-3),1.0,REVIEW_CTR,LOW_CTR_STRONG_POSITION,Very low evidence because impressions are extr...,More impressions or clicks could change the ob...
9,10,content_617fdc3f3a5d41eb,2,0,0.0,0.0,Strong (1-3),1.0,REVIEW_CTR,LOW_CTR_STRONG_POSITION,Very low evidence because impressions are extr...,More impressions or clicks could change the ob...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 4. Weak picks + leakage check

### Weak picks

Some of the top-ranked pages are weak picks because they have very few impressions. Although their CTR is 0% and their average position is strong, the evidence is limited because the pages received only 1–3 impressions.

This means the baseline score can rank pages highly even when the observed CTR is based on very little search activity. These pages should therefore be treated as low-confidence review candidates rather than confirmed refresh opportunities.

### Leakage check

The baseline rule uses only March 2026 search-performance signals: impressions, clicks, CTR, and average position.

No future-window data was used in the score. No product flags or future labels were used as inputs to the rule.

The output is therefore a directional decision-support baseline, not a prediction of a future outcome.

In [16]:
# Section 4 — Weak picks + leakage check

# Check the top-20 picks with very low impressions
weak_picks = top20[top20['gsc_impressions'] <= 3].copy()

print("Weak picks (impressions <= 3):")
display(
    weak_picks[
        [
            'rank',
            'content_hash_id',
            'gsc_impressions',
            'gsc_clicks',
            'ctr',
            'gsc_avg_position',
            'baseline_score',
            'reason_code',
            'action'
        ]
    ]
)

print("Number of weak picks:", len(weak_picks))


# Leakage check — confirm the queue only uses March 2026
print("\nLeakage checks")

print(
    "Start date:",
    ranked_queue['report_date'].min()
)

print(
    "End date:",
    ranked_queue['report_date'].max()
)

print(
    "Future dates present:",
    (ranked_queue['report_date'] > pd.Timestamp('2026-03-31')).any()
)


# Confirm the scoring inputs used by the baseline
scoring_inputs = [
    'gsc_impressions',
    'gsc_clicks',
    'ctr',
    'gsc_avg_position'
]

print(
    "Scoring inputs:",
    scoring_inputs
)

print(
    "No future-window features used: CONFIRMED"
)

print(
    "No product flags used as scoring inputs: CONFIRMED"
)

Weak picks (impressions <= 3):


,rank,content_hash_id,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,baseline_score,reason_code,action
0,1,content_34280304abbab05e,3,0,0.0,0.0,1.0,LOW_CTR_STRONG_POSITION,REVIEW_CTR
1,2,content_ea60d7cef612b29c,1,0,0.0,0.0,1.0,LOW_CTR_STRONG_POSITION,REVIEW_CTR
2,3,content_03940665a88bf663,1,0,0.0,0.0,1.0,LOW_CTR_STRONG_POSITION,REVIEW_CTR
3,4,content_05f82b2bad6817cc,1,0,0.0,0.0,1.0,LOW_CTR_STRONG_POSITION,REVIEW_CTR
4,5,content_05597932fe4da067,1,0,0.0,0.0,1.0,LOW_CTR_STRONG_POSITION,REVIEW_CTR
5,6,content_a36aa972b8edda8f,2,0,0.0,0.0,1.0,LOW_CTR_STRONG_POSITION,REVIEW_CTR
6,7,content_d720dde3701523c0,1,0,0.0,0.0,1.0,LOW_CTR_STRONG_POSITION,REVIEW_CTR
7,8,content_eee959b8c3faf5c2,1,0,0.0,0.0,1.0,LOW_CTR_STRONG_POSITION,REVIEW_CTR
8,9,content_1f380a642aed423b,1,0,0.0,0.0,1.0,LOW_CTR_STRONG_POSITION,REVIEW_CTR
9,10,content_617fdc3f3a5d41eb,2,0,0.0,0.0,1.0,LOW_CTR_STRONG_POSITION,REVIEW_CTR


Number of weak picks: 20

Leakage checks
Start date: 2026-03-01 00:00:00
End date: 2026-03-31 00:00:00
Future dates present: False
Scoring inputs: ['gsc_impressions', 'gsc_clicks', 'ctr', 'gsc_avg_position']
No future-window features used: CONFIRMED
No product flags used as scoring inputs: CONFIRMED


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.